# Sentiment Analysis
This notebook cleans tweet data and evaluates VADER and TextBlob sentiment models.

In [1]:
import textblob
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import os
import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
df = pd.read_csv(r'/dataFolder\train.csv', encoding="latin-1")
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [3]:
text = df["text"]
sentiment = df["sentiment"]

text[0]

' I`d have responded, if I were going'

In [4]:
# Check for nulls first
print(df["text"].isna().sum())

1


In [5]:
df.loc[df["text"].isna()]

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
314,fdb77c3752,NaN,NaN,neutral,night,31-45,Namibia,2540905,823000.0,3


In [6]:
df.shape

(27481, 10)

In [7]:
# Fix by dropping or filling
df["text"] = df["text"].fillna("")

In [8]:
print(df.shape)
print(df["text"].isna().sum())

(27481, 10)
0


## Text Preprocessing
Remove punctuation, stopwords, then lemmatize to reduce words to their base form.

In [9]:
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
def remove_punct(text):
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'(.)\1{2,}', r'\1', text)     # sooo -> so, lolll -> lol

    # expand contractions before stripping punctuation
    text = re.sub(r"n[`']t", " not", text)        # couldn't/couldn`t -> could not
    text = re.sub(r"[`']re", " are", text)
    text = re.sub(r"[`']ve", " have", text)
    text = re.sub(r"[`']ll", " will", text)
    text = re.sub(r"[`']d",  " would", text)
    text = re.sub(r"[`']s",  "", text)

    text_nopunct = "".join([char for char in text if char not in string.punctuation])
    text_nopunct = text_nopunct.lower()

    stop_words = set(stopwords.words('english'))
    tokens = re.split('\W+', text_nopunct)
    filtered = [w for w in tokens if w not in stop_words]

    return filtered

df['body_text_clean'] = df['text'].apply(lambda x: remove_punct(x))
df.head()

In [11]:
def lemmetize(text):
    wn = nltk.WordNetLemmatizer()
    lemmatized = []
    for i in text:
        lemm = wn.lemmatize(i)
        lemmatized.append(lemm)
    return lemmatized

df['body_text_full_clean'] = df['body_text_clean'].apply(lambda x: lemmetize(x))
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²),body_text_clean,body_text_full_clean
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60,"[, id, responded, going]","[, id, responded, going]"
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105,"[, sooo, sad, miss, san, diego]","[, sooo, sad, miss, san, diego]"
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18,"[boss, bullying]","[bos, bullying]"
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164,"[, interview, leave, alone]","[, interview, leave, alone]"
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26,"[, sons, couldnt, put, releases, already, bought]","[, son, couldnt, put, release, already, bought]"


### WordNet POS Tagging
Map each token to its WordNet POS tag (noun, verb, adjective, adverb) for better lemmatization.

In [ ]:
from nltk.corpus import wordnet

def get_wordnet_tag(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)

df["wordnet_tag"] = df["body_text_full_clean"].apply(lambda tokens: [get_wordnet_tag(w) for w in tokens])
df[["body_text_full_clean", "wordnet_tag"]].head()

In [12]:
import re

def strip_html(text):
    return re.sub(r"<[^>]+>", "", text)

df["no_html"] = df["text"].apply(strip_html)

## Sentiment Predictions
Run VADER and TextBlob on the raw tweet text, then compare predictions against true labels.

In [13]:
vader = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = vader.polarity_scores(text)['compound']
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["vader"] = df["no_html"].apply(lambda x: vader_sentiment(x))
df[["vader", "sentiment", "no_html"]]

,vader,sentiment,no_html
0,neutral,neutral,"I`d have responded, if I were going"
1,negative,negative,Sooo SAD I will miss you here in San Diego!!!
2,negative,negative,my boss is bullying me...
3,negative,negative,what interview! leave me alone
4,neutral,negative,"Sons of ****, why couldn`t they put them on t..."
...,...,...,...
27476,positive,negative,wish we could come see u on Denver husband l...
27477,positive,negative,I`ve wondered about rake to. The client has ...
27478,positive,positive,Yay good for both of you. Enjoy the break - y...
27479,positive,positive,But it was worth it ****.


In [14]:
def textblob_sentiment(text):
    score = TextBlob(text).sentiment.polarity

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["textblob_pred"] = df["no_html"].apply(textblob_sentiment)
df[["vader", "sentiment", "no_html", "textblob_pred"]]

,vader,sentiment,no_html,textblob_pred
0,neutral,neutral,"I`d have responded, if I were going",neutral
1,negative,negative,Sooo SAD I will miss you here in San Diego!!!,negative
2,negative,negative,my boss is bullying me...,neutral
3,negative,negative,what interview! leave me alone,neutral
4,neutral,negative,"Sons of ****, why couldn`t they put them on t...",neutral
...,...,...,...,...
27476,positive,negative,wish we could come see u on Denver husband l...,neutral
27477,positive,negative,I`ve wondered about rake to. The client has ...,positive
27478,positive,positive,Yay good for both of you. Enjoy the break - y...,positive
27479,positive,positive,But it was worth it ****.,positive


In [15]:
def evaluate_model(true_labels, predicted_labels, model_name="Model"):
    """
    Evaluates sentiment model performance.

    Parameters:
        true_labels      : the actual correct labels from your dataset
        predicted_labels : the labels your model predicted
        model_name       : name to display in the output (e.g. "VADER", "TextBlob")
    """

    accuracy  = accuracy_score(true_labels, predicted_labels)
    f1        = f1_score(true_labels, predicted_labels, average="weighted")
    precision = precision_score(true_labels, predicted_labels, average="weighted", zero_division=0)
    recall    = recall_score(true_labels, predicted_labels, average="weighted")

    print(f"--- {model_name} ---")
    print(f"Accuracy  : {accuracy:.2%}")
    print(f"F1 Score  : {f1:.2%}")
    print(f"Precision : {precision:.2%}")
    print(f"Recall    : {recall:.2%}")
    print()

evaluate_model(df["sentiment"], df["textblob_pred"], "TextBlob")
evaluate_model(df["sentiment"], df["vader"],    "VADER")

--- TextBlob ---
Accuracy  : 58.87%
F1 Score  : 58.12%
Precision : 60.15%
Recall    : 58.87%

--- VADER ---
Accuracy  : 63.13%
F1 Score  : 62.29%
Precision : 65.68%
Recall    : 63.13%



In [16]:
def count(model):
    for label in df["sentiment"].unique():
        correct = sum((df["sentiment"] == label) & (df[model] == label))
        total = sum(df["sentiment"] == label)

        print(f"{label} — {correct}/{total}")

print("Vader:")
count("vader")

print("TextBlob:")
count("textblob_pred")

Vader:
neutral — 5245/11118
negative — 4640/7781
positive — 7463/8582
TextBlob:
neutral — 5978/11118
negative — 3416/7781
positive — 6785/8582


In [17]:
df["Vader Compound"] = df["text"].apply(lambda x: vader.polarity_scores(x)["compound"])
print(df["Vader Compound"].value_counts().head(10))

Vader Compound
 0.0000    7210
 0.4404     518
 0.3612     363
-0.2960     362
 0.4019     327
 0.5719     293
 0.4215     271
-0.4767     251
 0.6369     225
-0.1531     222
Name: count, dtype: int64


In [18]:
df["TextBlob Compound"] = df["text"].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
print(df["TextBlob Compound"].value_counts().head(10))

TextBlob Compound
 0.00    9428
 0.50    1035
 0.20     735
 1.00     684
 0.80     600
-0.50     595
 0.25     574
 0.10     390
 0.70     386
 0.60     317
Name: count, dtype: int64


## Takeaway
Vader predicts too many zeros, expected with messy text from social media

Same with TextBlob (even more zeros with TextBlob)